# Notebook 01: Surface Water and Flood Mapping
## Integrating Sentinel-1 SAR and Sentinel-2 Optical Data

**CEOS Best Practice Guidance — Multiplatform/Multisensor Data for Water Science**  
**Part I: Terrestrial Water Cycle | Variable: Surface Water Extent**

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ceos-org/ceos-water-guidance/blob/main/notebooks/terrestrial/01_surface_water_flood_mapping.ipynb)
[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/ceos-org/ceos-water-guidance/main?filepath=notebooks/terrestrial/01_surface_water_flood_mapping.ipynb)

---

**Authors:** John Bolten (NASA GSFC), [contributor names]  
**Version:** 0.1-draft | **Last updated:** April 2026  
**License:** CC BY 4.0

---

## Learning Objectives

By the end of this notebook, you will be able to:

1. Explain **why SAR and optical data are complementary** for surface water mapping — and when each fails alone
2. Access **Sentinel-1 SAR** and **Sentinel-2 multispectral** data through open cloud APIs (Microsoft Planetary Computer / Copernicus Dataspace)
3. Apply **water indices** (NDWI, MNDWI) to optical data and **backscatter thresholding** to SAR data
4. **Fuse the two products** using a simple decision-tree approach to produce an all-weather surface water map
5. Quantify **uncertainty** in the fused product and understand where the method breaks down
6. Apply the workflow to a **flood event** to detect inundation extent

---

## 1. Science Background

### 1.1 Why Surface Water Matters

Surface water — rivers, lakes, wetlands, reservoirs, and floodplains — is both a critical resource and a natural hazard. Accurate, timely mapping of surface water extent is essential for:

- **Flood emergency response:** Where is water now? How fast is it spreading?
- **Water resource management:** How much water is stored in reservoirs and lakes?
- **Ecosystem monitoring:** Are wetlands expanding or contracting over time?
- **Agricultural planning:** Which fields are inundated and for how long?

### 1.2 Why Optical Alone is Not Enough

Optical sensors detect water using the strong absorption of liquid water in near-infrared and shortwave infrared wavelengths. Indices like NDWI and MNDWI reliably identify open water under clear-sky conditions.

**The problem:** Floods don't care about cloud cover. Major flooding events are almost always accompanied by persistent cloud cover. In tropical regions, cloud cover exceeds 70% on any given day. An optical sensor that can't see through clouds is least useful precisely when water mapping is most urgent.

| Sensor Type | Cloud Penetration | Spectral Info | Best For |
|---|---|---|---|
| Optical (Sentinel-2, Landsat) | ❌ None | ✅ Rich | Clear-sky water mapping, water quality |
| SAR (Sentinel-1) | ✅ Full | ❌ Limited | All-weather flood detection, soil moisture |
| **Fused Optical + SAR** | ✅ Full | ✅ Rich | **All-weather, all-season water mapping** |

### 1.3 How SAR Detects Water

SAR microwave pulses scattered by a smooth open water surface return almost no energy back to the sensor — producing a very **low backscatter signal** (dark areas in SAR imagery). This physical principle is robust and consistent across sensors and acquisitions.

**Limitations of SAR alone:**
- Wind roughens water surfaces, raising backscatter and causing missed detections
- Flooded vegetation (water under a canopy) can have *high* backscatter due to double-bounce
- Urban areas and steep terrain create geometric distortions
- SAR cannot distinguish water from other smooth surfaces (sand, compacted soil)

### 1.4 The Integration Rationale

The complementary nature of optical and SAR means that a fused approach outperforms either alone:

```
Optical available (clear sky)  →  Use MNDWI water mask (more accurate)
Optical unavailable (clouds)   →  Use SAR backscatter threshold (all-weather)
Both available                 →  Optical confirms SAR detections, reduces false positives
Both disagree                  →  Flag as uncertain, apply additional rules
```

> **Best Practice #1:** Never rely on a single acquisition date from a single sensor for flood mapping. Compute a pre-event baseline from both optical and SAR, then detect change. Change detection is more reliable than absolute thresholding.

---

## 2. Data Sources and Access

### 2.1 Sentinel-1 SAR (C-band)

- **Operator:** ESA / Copernicus Programme
- **Wavelength:** C-band (5.6 cm), VV and VH polarizations
- **Spatial resolution:** 10 m (IW mode GRD)
- **Revisit:** ~6 days (with Sentinel-1A + 1C)
- **CEOS-ARD:** Normalized Radar Backscatter (NRB) specification available at [ceos.org/ard](https://ceos.org/ard)
- **Access:** [Copernicus Dataspace](https://dataspace.copernicus.eu/) | [Microsoft Planetary Computer](https://planetarycomputer.microsoft.com/)

### 2.2 Sentinel-2 Multispectral (MSI)

- **Operator:** ESA / Copernicus Programme  
- **Bands used:** Green (B3, 560 nm), NIR (B8, 842 nm), SWIR1 (B11, 1610 nm)
- **Spatial resolution:** 10–20 m
- **Revisit:** ~5 days (with Sentinel-2A + 2B)
- **CEOS-ARD:** Surface Reflectance (SR) specification: use L2A products
- **Access:** [Copernicus Dataspace](https://dataspace.copernicus.eu/) | [Microsoft Planetary Computer](https://planetarycomputer.microsoft.com/) | [Google Earth Engine](https://earthengine.google.com/)

### 2.3 Case Study: 2022 Pakistan Floods

This notebook uses the catastrophic 2022 Pakistan monsoon flooding as its case study — one of the largest flood events in recorded history, affecting over 33 million people across Sindh and Balochistan provinces. Cloud cover during the event made optical-only mapping impossible for weeks, making it an ideal demonstration of why SAR integration is essential.

**Study area:** Sindh Province, Pakistan (25°N–28°N, 67°E–70°E)  
**Pre-event period:** June 2022  
**Flood peak:** August–September 2022

---

## 3. Environment Setup

In [ ]:
# Install required packages (run once — skip if already installed)
# Uncomment the line below if running on Google Colab
# !pip install pystac-client planetary-computer rioxarray geopandas matplotlib odc-stac stackstac

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import geopandas as gpd
import xarray as xr
import rioxarray
from shapely.geometry import box
import planetary_computer
import pystac_client
import odc.stac
import warnings
warnings.filterwarnings('ignore')

print("Environment ready.")

In [ ]:
# ─── Study Area: Sindh Province, Pakistan ───────────────────────────────────
# Bounding box: [min_lon, min_lat, max_lon, max_lat]
BBOX = [67.5, 25.5, 69.5, 27.5]
RESOLUTION = 20  # metres

# Time windows
PRE_EVENT  = ("2022-06-01", "2022-06-30")   # Baseline: before monsoon onset
PEAK_FLOOD = ("2022-08-25", "2022-09-15")   # Peak inundation

print(f"Study area: {BBOX}")
print(f"Pre-event window:  {PRE_EVENT[0]} to {PRE_EVENT[1]}")
print(f"Flood peak window: {PEAK_FLOOD[0]} to {PEAK_FLOOD[1]}")

## 4. Sentinel-1 SAR Processing

We use VV polarization from Sentinel-1 IW GRD products, accessed as CEOS-ARD compliant Normalized Radar Backscatter (NRB) through Microsoft Planetary Computer. VV is preferred for open water detection because smooth water surfaces produce very low VV backscatter (~-20 to -25 dB).

> **Best Practice #2:** Always use **CEOS-ARD compliant** SAR products (Normalized Radar Backscatter) rather than raw GRD data. NRB products have terrain flattening, radiometric calibration, and speckle filtering applied — making them directly comparable across dates and sensors.

In [ ]:
# ─── Connect to Microsoft Planetary Computer STAC API ────────────────────────
catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)

def search_sentinel1(bbox, time_range, polarization="vv"):
    """Search for Sentinel-1 RTC (terrain corrected) scenes."""
    search = catalog.search(
        collections=["sentinel-1-rtc"],
        bbox=bbox,
        datetime=f"{time_range[0]}/{time_range[1]}",
        query={"platform": {"in": ["sentinel-1a", "sentinel-1b"]}},
    )
    items = list(search.items())
    print(f"Found {len(items)} Sentinel-1 scenes for {time_range[0]} to {time_range[1]}")
    return items

s1_pre   = search_sentinel1(BBOX, PRE_EVENT)
s1_flood = search_sentinel1(BBOX, PEAK_FLOOD)

In [ ]:
# ─── Load SAR backscatter (VV polarization) and compute median composites ────
def load_sar_composite(items, bbox, resolution=RESOLUTION):
    """Load Sentinel-1 VV backscatter and compute temporal median."""
    ds = odc.stac.load(
        items,
        bands=["vv"],
        bbox=bbox,
        resolution=resolution,
        chunks={"time": 1, "x": 512, "y": 512},
    )
    # Convert from linear to dB
    vv_db = 10 * np.log10(ds["vv"].where(ds["vv"] > 0))
    # Temporal median composite to reduce speckle
    composite = vv_db.median(dim="time", skipna=True)
    return composite

sar_pre   = load_sar_composite(s1_pre,   BBOX)
sar_flood = load_sar_composite(s1_flood, BBOX)

print(f"SAR composite shape: {sar_pre.shape}")
print(f"VV backscatter range (pre-event): {float(sar_pre.min()):.1f} to {float(sar_pre.max()):.1f} dB")

In [ ]:
# ─── SAR-based Water Mask ─────────────────────────────────────────────────────
# Open water: VV backscatter below threshold (-16 dB is commonly used for C-band)
# Reference: Twele et al. (2016), Chini et al. (2017)

SAR_WATER_THRESHOLD_DB = -16.0  # dB — pixels below this are classified as water

# Change detection: flood pixels have much lower backscatter than pre-event
sar_change = sar_flood - sar_pre  # negative values indicate flooding
CHANGE_THRESHOLD_DB = -3.0       # dB change threshold

# Combined SAR water mask
sar_water_mask = (
    (sar_flood < SAR_WATER_THRESHOLD_DB) |     # Absolute threshold
    (sar_change < CHANGE_THRESHOLD_DB)         # Change detection
).compute()

sar_water_fraction = float(sar_water_mask.sum()) / float(sar_water_mask.size) * 100
print(f"SAR water mask: {sar_water_fraction:.1f}% of pixels classified as water")

# ─── Visualize ───────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].imshow(sar_pre, cmap='gray', vmin=-25, vmax=0)
axes[0].set_title("SAR VV: Pre-event (June 2022)", fontsize=11)
axes[0].axis('off')

axes[1].imshow(sar_flood, cmap='gray', vmin=-25, vmax=0)
axes[1].set_title("SAR VV: Peak flood (Aug-Sep 2022)", fontsize=11)
axes[1].axis('off')

axes[2].imshow(sar_water_mask, cmap='Blues', vmin=0, vmax=1)
axes[2].set_title("SAR Water Mask (combined threshold)", fontsize=11)
axes[2].axis('off')

plt.suptitle("Sentinel-1 SAR: Surface Water Detection — Pakistan Floods 2022",
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig("outputs/sar_water_mask.png", dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved to outputs/sar_water_mask.png")

## 5. Sentinel-2 Optical Processing

We compute the Modified Normalized Difference Water Index (MNDWI) from Sentinel-2 L2A Surface Reflectance data. MNDWI uses the green and SWIR1 bands to maximize sensitivity to open water while suppressing urban and vegetation false positives.

$$\text{MNDWI} = \frac{\rho_{Green} - \rho_{SWIR1}}{\rho_{Green} + \rho_{SWIR1}}$$

> **Best Practice #3:** Use MNDWI rather than NDWI for mapping water in agricultural and urban areas. MNDWI (Green−SWIR1) has lower commission errors in built-up environments than NDWI (Green−NIR). For densely vegetated wetlands, also compute AWEIsh as a secondary confirmation.

> **Best Practice #4:** Only use Sentinel-2 pixels with **Scene Classification Layer (SCL)** values indicating clear sky (SCL = 4: vegetation, 5: bare soil, 6: water). Exclude cloud (SCL 8,9,10), cloud shadow (SCL 3), and saturated pixels (SCL 1).

In [ ]:
# ─── Search and Load Sentinel-2 L2A Data ─────────────────────────────────────
def search_sentinel2(bbox, time_range, max_cloud_cover=30):
    """Search for Sentinel-2 L2A scenes below cloud cover threshold."""
    search = catalog.search(
        collections=["sentinel-2-l2a"],
        bbox=bbox,
        datetime=f"{time_range[0]}/{time_range[1]}",
        query={"eo:cloud_cover": {"lt": max_cloud_cover}},
    )
    items = list(search.items())
    print(f"Found {len(items)} Sentinel-2 scenes (cloud cover < {max_cloud_cover}%) "
          f"for {time_range[0]} to {time_range[1]}")
    return items

# NOTE: during peak flooding, cloud cover is high — we may find very few clear scenes
# This is exactly the scenario where SAR integration becomes critical
s2_pre   = search_sentinel2(BBOX, PRE_EVENT,  max_cloud_cover=30)
s2_flood = search_sentinel2(BBOX, PEAK_FLOOD, max_cloud_cover=80)  # relax threshold during flood

print(f"\nNote: Fewer clear optical scenes during flooding → SAR is essential here")

In [ ]:
# ─── Compute MNDWI and Cloud-masked Water Index ───────────────────────────────
def compute_mndwi_composite(items, bbox, resolution=RESOLUTION):
    """Compute cloud-masked MNDWI composite from Sentinel-2 L2A."""
    ds = odc.stac.load(
        items,
        bands=["B03", "B11", "SCL"],  # Green, SWIR1, Scene Classification
        bbox=bbox,
        resolution=resolution,
        chunks={"time": 1, "x": 512, "y": 512},
    )
    # Scale surface reflectance (L2A values are scaled by 10000)
    green = ds["B03"].astype(float) / 10000.0
    swir1 = ds["B11"].astype(float) / 10000.0
    scl   = ds["SCL"]

    # Cloud mask: keep only clear-sky pixels
    # SCL classes 4 (veg), 5 (bare soil), 6 (water) are clear
    clear_mask = scl.isin([4, 5, 6])

    # Compute MNDWI
    mndwi = ((green - swir1) / (green + swir1 + 1e-10)).where(clear_mask)

    # Temporal median composite (only clear pixels contribute)
    mndwi_composite = mndwi.median(dim="time", skipna=True)
    clear_coverage  = clear_mask.any(dim="time").compute()  # pixels with any clear obs

    return mndwi_composite.compute(), clear_coverage

mndwi_pre,   clear_pre   = compute_mndwi_composite(s2_pre,   BBOX)
mndwi_flood, clear_flood = compute_mndwi_composite(s2_flood, BBOX)

optical_coverage_pct = float(clear_flood.sum()) / float(clear_flood.size) * 100
print(f"Optical clear-sky coverage during flood: {optical_coverage_pct:.0f}% of pixels")
print(f"→ {100-optical_coverage_pct:.0f}% of area needs SAR to fill the gap")

In [ ]:
# ─── Optical Water Mask from MNDWI ───────────────────────────────────────────
# MNDWI > 0 is the standard threshold for open water; use 0.1 to reduce false positives
MNDWI_WATER_THRESHOLD = 0.1

optical_water_mask = (mndwi_flood > MNDWI_WATER_THRESHOLD).where(clear_flood, other=np.nan)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(mndwi_flood, cmap='RdYlBu', vmin=-0.5, vmax=0.8)
axes[0].set_title("MNDWI: Peak flood period\n(cloud-masked — grey = no data)", fontsize=11)
axes[0].axis('off')

cmap_water = mcolors.ListedColormap(['lightyellow', 'steelblue'])
axes[1].imshow(optical_water_mask, cmap=cmap_water, vmin=0, vmax=1)
axes[1].set_title("Optical water mask (MNDWI > 0.1)\nGrey = cloud-covered, no data", fontsize=11)
axes[1].axis('off')

plt.suptitle("Sentinel-2: Surface Water from MNDWI — Pakistan Floods 2022",
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig("outputs/optical_water_mask.png", dpi=150, bbox_inches='tight')
plt.show()

## 6. Optical + SAR Fusion

The fusion strategy is a **decision-tree** approach: use the most reliable data source available for each pixel, and flag uncertainty where sources disagree.

```
For each pixel:
  IF optical data available (clear sky):
      Use optical classification (more spectrally informative)
      IF SAR also available AND disagrees:
          Flag as LOW CONFIDENCE
  ELSE (cloud covered):
      Use SAR classification
      Mark as SAR-ONLY (moderate confidence)
  IF neither available:
      Mark as NO DATA
```

> **Best Practice #5:** Always produce a **confidence layer** alongside your water mask. At minimum, distinguish between: (1) optical-confirmed water, (2) SAR-only water, (3) optical-confirmed non-water, (4) SAR-only non-water, (5) conflicting sources, (6) no data. This allows downstream users to apply their own uncertainty thresholds.

In [ ]:
# ─── Fused Water Map with Confidence Classes ─────────────────────────────────
#
# Output classes:
#   0 = No data (neither optical nor SAR available)
#   1 = Optical + SAR agree: DRY (high confidence non-water)
#   2 = Optical + SAR agree: WATER (high confidence water)
#   3 = Optical says DRY, SAR says WATER (check: wind roughening? flooded vegetation?)
#   4 = Optical says WATER, SAR says DRY (check: calm open water? turbid water?)
#   5 = SAR only: WATER (optical cloud-covered)
#   6 = SAR only: DRY (optical cloud-covered)
#   7 = Optical only: WATER (SAR data gap)
#   8 = Optical only: DRY (SAR data gap)

# Boolean masks
opt_available = clear_flood                                         # True where optical is clear
opt_water     = (mndwi_flood > MNDWI_WATER_THRESHOLD) & opt_available
opt_dry       = (mndwi_flood <= MNDWI_WATER_THRESHOLD) & opt_available
sar_water     = sar_water_mask
sar_dry       = ~sar_water_mask

fused = xr.zeros_like(sar_water.astype(int))
fused = fused.where(~(opt_available & sar_water  & opt_water), other=2)  # Both water
fused = fused.where(~(opt_available & ~sar_water & opt_dry),   other=1)  # Both dry
fused = fused.where(~(opt_available & sar_water  & opt_dry),   other=3)  # SAR water, opt dry
fused = fused.where(~(opt_available & ~sar_water & opt_water), other=4)  # Opt water, SAR dry
fused = fused.where(~(~opt_available & sar_water),             other=5)  # SAR only water
fused = fused.where(~(~opt_available & ~sar_water),            other=6)  # SAR only dry

# Summary statistics
total = fused.size
for cls, label in [(2, 'High-conf water (both agree)'),
                   (5, 'SAR-only water (cloud covered)'),
                   (3, 'Conflict: SAR water, optical dry'),
                   (4, 'Conflict: optical water, SAR dry')]:
    count = int((fused == cls).sum())
    print(f"  Class {cls} — {label}: {count/total*100:.1f}%")

In [ ]:
# ─── Final Map ───────────────────────────────────────────────────────────────
colors = {
    0: '#DDDDDD',  # No data — light grey
    1: '#F5E6C8',  # High-conf dry — pale sand
    2: '#1565C0',  # High-conf water — deep blue
    3: '#FFA726',  # Conflict: SAR water, opt dry — orange
    4: '#AB47BC',  # Conflict: opt water, SAR dry — purple
    5: '#42A5F5',  # SAR-only water — light blue
    6: '#BCAAA4',  # SAR-only dry — warm grey
}
cmap = mcolors.ListedColormap([colors[i] for i in sorted(colors.keys())])

fig, ax = plt.subplots(1, 1, figsize=(10, 8))
im = ax.imshow(fused, cmap=cmap, vmin=0, vmax=8, interpolation='none')
ax.set_title("Fused Optical + SAR Surface Water Map\nPakistan Floods — Peak Event, Aug-Sep 2022",
             fontsize=13, fontweight='bold')
ax.axis('off')

# Legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#1565C0', label='Water — optical + SAR confirmed'),
    Patch(facecolor='#42A5F5', label='Water — SAR only (cloud covered)'),
    Patch(facecolor='#F5E6C8', label='Dry — optical + SAR confirmed'),
    Patch(facecolor='#FFA726', label='Uncertain — SAR water, optical dry'),
    Patch(facecolor='#AB47BC', label='Uncertain — optical water, SAR dry'),
    Patch(facecolor='#DDDDDD', label='No data'),
]
ax.legend(handles=legend_elements, loc='lower left', fontsize=9, framealpha=0.9)
plt.tight_layout()
plt.savefig("outputs/fused_water_map.png", dpi=150, bbox_inches='tight')
plt.show()
print("Final fused map saved to outputs/fused_water_map.png")

## 7. Uncertainty and Known Limitations

No surface water mapping approach is error-free. Understanding where and why the method fails is essential for responsible use.

In [ ]:
# ─── Quantify Conflict Zones ──────────────────────────────────────────────────
conflict_fraction = float(((fused == 3) | (fused == 4)).sum()) / total * 100
sar_only_fraction = float((fused == 5).sum()) / total * 100
nodata_fraction   = float((fused == 0).sum()) / total * 100

print("=== Uncertainty Summary ===")
print(f"SAR-only coverage (cloud-filled):   {sar_only_fraction:.1f}% of study area")
print(f"Conflicting source classifications: {conflict_fraction:.1f}% of study area")
print(f"No data (neither source):           {nodata_fraction:.1f}% of study area")
print()
print("Known sources of error in this approach:")
print("  SAR commission errors:  Wind-roughened water misclassified as dry")
print("  SAR omission errors:    Flooded vegetation under canopy (high double-bounce)")
print("  Optical commission:     Cloud shadow misclassified as water")
print("  Optical omission:       Turbid/sediment-laden water (low MNDWI)")
print()
print("> Best Practice #6: Report conflict fraction and SAR-only fraction as")
print("  metadata with any derived flood statistics.")

## 8. Validation Against In-Situ and Reference Data

> **Best Practice #7:** Validate your water mask against independent reference data before using it in any operational or decision-making context. Suitable reference sources include:
> - **JRC Global Surface Water** (optical-based, good for clear conditions, baseline reference)
> - **UNOSAT flood maps** (expert-derived SAR maps for major events)
> - **USGS/national gauge station flood records**
> - **Field survey data** where available

A full validation is beyond the scope of this notebook, but we demonstrate a comparison against the JRC Global Surface Water product as a sanity check.

In [ ]:
# ─── Compare against JRC Global Surface Water ────────────────────────────────
# JRC GSW provides historical water occurrence maps from 1984-present
# We use the 'occurrence' band: percentage of observations where water was detected
# A pixel with occurrence > 75% is considered 'permanent water'

jrc_search = catalog.search(
    collections=["jrc-gsw"],
    bbox=BBOX,
)
jrc_items = list(jrc_search.items())

if jrc_items:
    jrc_ds = odc.stac.load(
        jrc_items, bands=["occurrence"], bbox=BBOX, resolution=RESOLUTION
    )
    jrc_permanent = (jrc_ds["occurrence"].squeeze() > 75).compute()

    # Simple agreement check
    fused_water = (fused == 2) | (fused == 5)  # All water pixels
    agreement = float((fused_water == jrc_permanent).sum()) / total * 100
    print(f"Pixel-level agreement with JRC permanent water: {agreement:.1f}%")
    print("Note: flood extent > permanent water is expected during the 2022 event")
else:
    print("JRC GSW not available through this endpoint — use GEE or direct download")
    print("Reference: Pekel et al. (2016), Nature, doi:10.1038/nature20584")

## 9. Best Practice Summary

This notebook demonstrated the following best practices for optical + SAR surface water mapping:

| # | Best Practice | Why It Matters |
|---|---|---|
| 1 | Use change detection, not just absolute thresholds | More robust to inter-acquisition variation in SAR backscatter |
| 2 | Use CEOS-ARD compliant SAR products (NRB) | Enables direct comparison across dates and sensors |
| 3 | Prefer MNDWI over NDWI in mixed land-cover | Reduces false positives in urban and agricultural areas |
| 4 | Apply SCL cloud masking to Sentinel-2 | Essential — cloudy pixels corrupt water indices |
| 5 | Produce a confidence layer, not just a binary mask | Allows users to apply their own risk thresholds |
| 6 | Report conflict fraction and SAR-only fraction | Transparency about uncertainty in derived products |
| 7 | Validate against independent reference data | Required before any operational or decision-making use |

---

## 10. What to Explore Next

- **Flooded vegetation detection:** C-band SAR cross-polarization (VH/VV ratio) is more sensitive to double-bounce from flooded forests and rice paddies. See [notebook 01b — flooded vegetation] *(coming soon)*
- **Temporal dynamics:** Apply this workflow across a full monsoon season to produce inundation duration maps
- **SWOT integration:** Once SWOT data is available for your area, add surface water elevation to estimate storage volume
- **Soil moisture context:** See [Notebook 02](./02_soil_moisture_downscaling.ipynb) — soil saturation before flooding is a strong predictor of flood extent

---

## References

- Chini, M., et al. (2017). Sentinel-1 InSAR coherence to detect floodwater in urban areas. *Remote Sensing of Environment*, 187, 1–14.
- Manjusree, P., et al. (2012). Optimization of threshold ranges for rapid flood inundation mapping. *Asian Journal of Geoinformatics*.
- Pekel, J.-F., et al. (2016). High-resolution mapping of global surface water. *Nature*, 540, 418–422.
- Sentinel-1 NRB CEOS-ARD specification: https://ceos.org/ard/
- Twele, A., et al. (2016). Sentinel-1-based flood mapping: a fully automated processing chain. *International Journal of Remote Sensing*, 37(13), 2990–3004.
- Xu, H. (2006). Modification of normalised difference water index to enhance open water features. *International Journal of Remote Sensing*, 27(14), 3025–3033.

---

## Contributing to This Notebook

Found a bug? Know a better threshold? Have a regional adaptation?  
→ [Open an issue](https://github.com/ceos-org/ceos-water-guidance/issues)  
→ [Submit a pull request](https://github.com/ceos-org/ceos-water-guidance/blob/main/CONTRIBUTING.md)

*All contributors are credited in the Zenodo DOI record.*